# KG1 — EVAL 086 + candidato no full947 (Claude) — Rota B
**Run all.** Colab A100 + Secret HF_KEY. Mede ACC REAL (métrica oficial vLLM, rank<=32) do 086 e de um candidato no full947 canônico (947). 086 (0.86)=piso. Para um candidato diverso, defina `os.environ['CAND_REPO']` antes do run.

In [ ]:
import os,subprocess,sys
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'],check=True)
os.chdir('/content/kg1')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','vllm==0.19.1','safetensors','huggingface_hub','hf_xet'],check=False)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'],check=False)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'],check=False)
print('DEPS OK',flush=True)


In [ ]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'),'Defina HF_KEY no Colab Secrets'
print('HF token OK',flush=True)


In [ ]:
import os,subprocess,sys,time
os.chdir('/content/kg1')
from huggingface_hub import snapshot_download
SOL='/content/kg1/artifacts/v1244_cot_safe/full947_solution.csv'  # canonico 947 (metrica oficial)
BASE=('086','felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086','f4134a6d223249d27be2f1c5d94ed59d118d1ce5')
CAND=('CAND', os.environ.get('CAND_REPO','felipesp1983/kg1-v1244-cot-candidate'), os.environ.get('CAND_REV',''))
# --- LIVE-LOG: stream eval -> HF kg1-live-logs (Claude monitora AO VIVO) ---
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','1')  # eval longo: pode trocar p/ '0' se hiccup de upload nao puder abortar
for tag,repo,rev in [BASE,CAND]:
    try:
        local=snapshot_download(repo_id=repo, revision=(rev or None))
    except Exception as e:
        print('SKIP',tag,repo,'(',str(e)[:60],')',flush=True); continue
    out='/content/eval_'+tag
    os.environ['RUN_ID']='v1244_eval_'+tag+'_'+time.strftime('%Y%m%d_%H%M%S')
    print('LIVE RUN_ID=',os.environ['RUN_ID'],'-> RUN',tag,flush=True)
    cmd=[sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/evaluate_lora_adapter.py','--solution-csv',SOL,'--adapter',local,'--output-dir',out,'--label',tag]
    subprocess.run(cmd)
print('Compare /content/eval_086 vs /content/eval_CAND (per_task.csv): +bit,+eq,0 regressao protegida, total>=843.',flush=True)
